# 01. E-commerce checkout: plan, measure, and reduce variance

**Sector:** e-commerce electronics. **Decision:** is the new one-click checkout
worth launching? We want the effect on **revenue per user**.

This case walks through a complete online A/B flow:

1. **Plan** the sample size (`power_analysis`) from the historical revenue
   variance.
2. **Design** with complete randomization (`CRD`) and **check SRM** before
   looking at any result.
3. **Estimate** the effect with the Neyman variance (finite population) and
   compare with the bootstrap (superpopulation).
4. **Reduce variance** with `CUPED`, using pre-experiment spend as the
   covariate.

Theory: [I. Foundations](../../../docs/en/theory/01-foundations.md),
[IV. Inference](../../../docs/en/theory/04-inference.md) (topics 14, 15 and 17)
and [III. Estimation](../../../docs/en/theory/03-estimation.md) (topic 11, CUPED).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "ecommerce_checkout.csv")
print(df.shape)
df.head()


## 1. How many users? (power)

Revenue per user has a long tail (a few users spend a lot), so its variance is
high. We use the historical standard deviation (the `revenue_y0` column, the
no-treatment behavior) to size the test for a minimum detectable effect (MDE) of
`+2.5` in revenue, with 80% power and `alpha = 0.05`.


In [ ]:
from skxperiments.design.power import power_analysis

std_hist = float(df["revenue_y0"].std())
plan = power_analysis(mde=2.5, power=0.8, std=std_hist, alpha=0.05)
print(f"historical std: {std_hist:.1f}")
print(f"required n_total: {plan.n_total}  (~{plan.n_treated} per arm)")
print(f"we collected {len(df)} users, comfortably above the plan")


In [ ]:
from skxperiments.reporting import plot_power_curve

n_values = list(range(200, 4001, 200))
ax = plot_power_curve(n_values, mde=2.5, std=std_hist, alpha=0.05)
ax.set_title("Power vs. sample size (MDE = 2.5)")
ax.figure


## 2. Design and SRM check

We randomize users 50/50 with `CRD`. Then we build the observed outcome: each
user reveals `revenue_y1` if assigned to treatment and `revenue_y0` otherwise
(the "science table" of the synthetic data). Before estimating anything, we run
the `SRMTest`: if the observed proportion does not match the planned one,
something broke and nothing else is trustworthy.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest

design = CRD(p=0.5, seed=101)
assignment = design.randomize(df[["device", "sessions_pre", "spend_pre"]].copy())

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["revenue"] = np.where(t == 1, df["revenue_y1"].to_numpy(), df["revenue_y0"].to_numpy())
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=101
)

srm = SRMTest().run(assignment)
print(f"SRM flagged: {srm.flagged}  (p={srm.p_value:.3f})")


## 3. Estimate the effect and reduce the variance

- `DifferenceInMeans` + `NeymanCI`: the point and confidence interval in the
  **finite-population** view (SATE).
- `BootstrapCI`: the **superpopulation** view (PATE), resampling within each arm.
- `CUPED`: uses pre-experiment spend (`spend_pre`), correlated with revenue, to
  remove noise. The standard error drops without introducing bias.


In [ ]:
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.estimators.cuped import CUPED
from skxperiments.inference import NeymanCI, BootstrapCI

neyman = NeymanCI(estimator=DifferenceInMeans(outcome_col="revenue")).fit(assignment).estimate()
boot = BootstrapCI(
    estimator=DifferenceInMeans(outcome_col="revenue"), n_resamples=1000, seed=0
).fit(assignment).estimate()
cuped = BootstrapCI(
    estimator=CUPED(outcome_col="revenue", pre_experiment_col="spend_pre"),
    n_resamples=1000, seed=0,
).fit(assignment).estimate()

pd.DataFrame([
    {"method": "DIM / Neyman (SATE)", "ATE": neyman.ate, "SE": neyman.se,
     "CI_low": neyman.ci[0], "CI_high": neyman.ci[1]},
    {"method": "DIM / Bootstrap (PATE)", "ATE": boot.ate, "SE": boot.se,
     "CI_low": boot.ci[0], "CI_high": boot.ci[1]},
    {"method": "CUPED / Bootstrap", "ATE": cuped.ate, "SE": cuped.se,
     "CI_low": cuped.ci[0], "CI_high": cuped.ci[1]},
]).round(3)


In [ ]:
from skxperiments.reporting import plot_effect

ax = plot_effect(neyman)
ax.set_title("Revenue lift (Neyman 95% CI)")
ax.figure


## Decision

The true effect baked into the data is `+2.5`. All three approaches agree on the
point (close to `+2.5`), and the interval excludes zero: the new checkout
**increases revenue**. `CUPED` delivers the **smallest standard error**
(pre-experiment spend explains much of the revenue variance), so it is the most
precise reading to support the launch decision.

Natural next step: [02. Stores with blocking](02_fashion_stores_blocking.ipynb).
